In [ ]:
# ---------- Libraries ----------
#
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import time

from matplotlib.patches import Rectangle

from heimdall.config import PATH_DATASETS
from heimdall.data import DATASET_PADDING, DATASET_REGION, DATASET_SHAPES, DATASET_VARIABLES, MODALITIES
from heimdall.data.dataloader import get_dataloaders
from heimdall.data.dataset import HeimdallDataset
from heimdall.data.tools import count_halvings, crop, decode, open_zarr

# ---------- Jupyter ----------
#
%matplotlib inline
plt.rcParams.update({"font.size": 13})

# Making sure modules are reloaded when modified
%reload_ext autoreload
%autoreload 2

: 

### Encoders
Architecture of each encoder, from its configuration in `experiments/forecast-latent-states/configs/encoder/`, with its number of parameters (independent of the size of the grid).

In [ ]:
import math
import heimdall

from pathlib import Path
from shaggy.models.cae import ConvEncoder

from heimdall.tools import load_configuration

PATH_CONFIGS = Path(heimdall.__file__).parents[1] / "experiments" / "forecast-latent-states" / "configs" / "encoder"

print(f"{'config':<20} | {'levels':>6} | {'scale':<13} | {'latent':<22} | {'tokens':>7} | {'parameters':>10}")
for path in sorted(PATH_CONFIGS.glob("encoder-*.yml")):
    model = dict(load_configuration(path)[0]["Model"])
    modality, lat_channels = model.pop("modality"), model.pop("lat_channels")
    shape, variables = DATASET_SHAPES[modality], len(DATASET_VARIABLES[modality])

    # The inputs of Encoder are the fields, the observed points (observations only), the date, the land/sea mask and the mesh
    in_channels = (variables if modality == "glorys" else 2 * variables) + 2 + 1 + 2 * len(shape)
    encoder = ConvEncoder(in_channels, lat_channels, spatial=len(shape), **model)

    latent = [n // s for n, s in zip(shape, encoder.scale)]
    parameters = sum(p.numel() for p in encoder.parameters()) / 1e6
    print(f"{path.stem:<20} | {len(model['hid_channels']):>6} | {str(encoder.scale):<13} | {' x '.join(map(str, [lat_channels, *latent])):<22} | {math.prod(latent):>7} | {parameters:>8.1f} M")
